# Evaluation Metrics for Chart/Document VQA

This notebook walks through the three metrics in `production_vlm.eval` that replace BLEU and exact-match for chart/document visual QA: `numeric_accuracy`, `grounding_score`, and `faithfulness_score`. It shows why standard text metrics fail and how these alternatives work.

References:
- Es et al. (2023) *RAGAS: Automated Evaluation of Retrieval Augmented Generation* (arXiv:2309.15217) — adapted here from retrieved-text to chart/image evidence
- Hendrycks & Dietterich (2019) *Benchmarking Neural Network Robustness to Common Corruptions* (arXiv:1903.12261)

In [1]:
# Setup — works with the CPU-only core (no GPU, no torch required)
import sys
sys.path.insert(0, '../src')  # adjust if running from repo root

from production_vlm.eval import numeric_accuracy, grounding_score, faithfulness_score
from production_vlm.utils.synthetic_charts import generate_synthetic_chart
import numpy as np

print("production_vlm.eval imported successfully")

production_vlm.eval imported successfully


## Why BLEU and exact-match fail on chart answers

Consider a model that correctly reads a chart showing 4.2M revenue.
The prediction and reference are semantically identical but formatted differently.

In [2]:
prediction = "The chart shows revenue of $4.21M"
reference  = "Revenue was 4.2 million dollars"

# BLEU would score this near 0 — no overlapping n-grams
# Exact-match would return False — different strings
# But the numeric answer IS correct within 0.5%

result = numeric_accuracy(prediction, reference, tolerance=0.02)
print(f"Score:     {result.score:.1f}  (1.0 = correct)")
print(f"Matched:   {result.matched}/{result.total_reference} numbers")
print(f"Tolerance: ±{result.tolerance:.0%} relative")
print()
print("Compare: BLEU would score ≈ 0.0 here (no n-gram overlap)")

Score:     1.0  (1.0 = correct)
Matched:   1/1 numbers
Tolerance: ±2% relative

Compare: BLEU would score ≈ 0.0 here (no n-gram overlap)


## Numeric accuracy: partial credit and tolerance

The metric extracts numbers, handles commas and percent signs, and matches them greedily:

In [3]:
# Partial match: 2 correct out of 3 reference numbers
result = numeric_accuracy(
    prediction="North: 50 units, South: 67.6, West: 999",
    reference="North: 50.0 units, South: 67.6 units, East: 30.2 units",
)
print(f"Score:   {result.score:.3f}  ({result.matched}/{result.total_reference} matched)")
print()

# Tolerance demonstration
for tol in [0.001, 0.01, 0.02, 0.05]:
    r = numeric_accuracy("revenue 4.22", "revenue 4.20", tolerance=tol)
    print(f"  tolerance={tol:.1%}: score={r.score:.0f}  (|4.22-4.20|/4.20 = {abs(4.22-4.20)/4.20:.1%})")

Score:   0.667  (2/3 matched)

  tolerance=0.1%: score=0  (|4.22-4.20|/4.20 = 0.5%)
  tolerance=1.0%: score=0  (|4.22-4.20|/4.20 = 0.5%)
  tolerance=2.0%: score=1  (|4.22-4.20|/4.20 = 0.5%)
  tolerance=5.0%: score=1  (|4.22-4.20|/4.20 = 0.5%)


## Grounding score: is the answer actually referencing the chart?

A model can give a fluent, plausible answer that has no grounding in the actual chart values. `grounding_score` catches this:

In [4]:
evidence = "South: 67.6 req/s; North: 50.0 req/s; East: 45.2 req/s"

# Grounded answer
r_grounded = grounding_score(
    prediction="South region showed throughput of 67.6 req/s, the highest",
    evidence_text=evidence,
)
print(f"Grounded answer:    {r_grounded.score:.2f}  ({r_grounded.grounded_terms}/{r_grounded.total_terms} terms)")

# Hallucinated answer — fluent but not from the chart
r_hallucinated = grounding_score(
    prediction="The metrics indicate a moderate baseline across all regions",
    evidence_text=evidence,
)
print(f"Hallucinated answer: {r_hallucinated.score:.2f}  ({r_hallucinated.grounded_terms}/{r_hallucinated.total_terms} terms)")

Grounded answer:    0.71  (5/7 terms)
Hallucinated answer: 0.00  (0/5 terms)


## Composite faithfulness score on real synthetic charts

`faithfulness_score` combines both metrics with configurable weights (default: 60% numeric, 40% grounding). Here we score a correct answer, a vague answer, and a hallucinated answer against a real synthetic chart:

In [5]:
chart = generate_synthetic_chart(seed=5, render_image=False)
print(f"Chart: {chart.title}")
print(f"Evidence: {chart.evidence_text}")
print(f"Reference answer: {chart.answer}")
print()

predictions = {
    "Correct (paraphrased)": chart.answer.replace("has", "shows").replace("which is", "making it"),
    "Vague / ungrounded":    "The values appear moderate based on the chart data.",
    "Wrong numbers":         f"{chart.categories[0]} has a value of {chart.values[-1]*3:.1f} {chart.units}, the highest.",
}

print(f"{'Prediction':<28} {'Faithfulness':>14} {'Numeric':>9} {'Grounding':>11}")
print("-" * 65)
for label, pred in predictions.items():
    r = faithfulness_score(pred, chart.answer, chart.evidence_text)
    print(f"{label:<28} {r.score:>14.3f} {r.numeric.score:>9.3f} {r.grounding.score:>11.3f}")

Chart: Conversion Rate by Quarter
Evidence: Q2: 97.1 %; Q3: 97.8 %; Q1: 89.3 %; Q4: 74.4 %
Reference answer: Q3 shows conversion rate of 97.8 %, making it the highest.

Prediction                   Faithfulness    Numeric   Grounding
-----------------------------------------------------------------
Correct (paraphrased)               0.720     0.800       0.600
Vague / ungrounded                  0.000     0.000       0.000
Wrong numbers                       0.200     0.000       0.500


## Key takeaway

| Metric | Right tool when | Wrong tool when |
|---|---|---|
| `numeric_accuracy` | Answer is a number or contains numbers | Answer is qualitative / no numbers in reference |
| `grounding_score` | Checking if the answer references real evidence terms | Evidence has very short or stop-word-dominated vocabulary |
| `faithfulness_score` | Single ranking scalar for training signal or A/B comparison | You need fine-grained per-dimension analysis (use the sub-metrics directly) |

The `ran_with_real_ml_stack` flag in `results.json` tells you whether the numbers above came from a real VLM or the CPU smoke-test proxy. These metrics evaluate *whatever output your model gives* — they don't care whether it came from a 2B or 70B VLM.